# Sales Analytics Walkthrough

This notebook builds the same analysis as the project's dashboard, but
step-by-step so you can see what each stage produces along the way.

**What we're doing, in plain English:** we have a fictional online store's
sales records. We'll (1) generate that fake data, (2) organize it into a
proper database, (3) look at basic KPIs (revenue, orders, etc.), (4) predict
next 6 months of revenue, and (5) group customers by buying behavior.

Run each cell below in order (click a cell, press **Shift+Enter**).


In [1]:
# Make sure Python can find the project's src/ folder, no matter where
# this notebook is opened from.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # if opened from inside notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import sqlite3
import plotly.express as px

pd.set_option("display.max_columns", None)
print(f"Project root: {PROJECT_ROOT}")


Project root: /home/user/sales-analytics-dashboard


## Step 1: Generate the sales data

This invents ~12,000 realistic order line items — fake customers, products,
dates, and discounts — and saves them as a CSV. It's random but repeatable
(same data every time you run it).


In [2]:
import generate_sales_data
generate_sales_data.main()


Wrote 12,000 rows to /home/user/sales-analytics-dashboard/data/raw/sales_data.csv
Date range: 2023-01-01 to 2025-12-31
Unique customers: 817
Unique orders: 7,049


Let's look at a few rows of what got created:

In [3]:
sales_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "sales_data.csv")
sales_df.head()


,order_id,order_date,customer_id,region,product_name,category,unit_price,quantity,discount_pct,gross_amount,discount_amount,net_amount
0,ORD-000589,2023-01-01,CUST-00811,North America,Quill Leather Notebook,Office Supplies,19.99,1,0.00,19.99,0.00,19.99
1,ORD-000589,2023-01-01,CUST-00811,North America,Ember Scented Candle Set,Home Goods,24.99,3,0.05,74.97,3.75,71.22
2,ORD-003943,2023-01-01,CUST-00656,North America,Comet Office Chair,Furniture,219.99,1,0.20,219.99,44.00,175.99
3,ORD-003943,2023-01-01,CUST-00656,North America,Ember Scented Candle Set,Home Goods,24.99,1,0.10,24.99,2.50,22.49
4,ORD-004189,2023-01-01,CUST-00085,Europe,Haven Bookshelf,Furniture,159.99,1,0.05,159.99,8.00,151.99


## Step 2: Load it into a database

A flat CSV isn't how real companies store sales data — they use a
**relational database** with separate tables for customers, products, and
orders, linked by IDs. This step reorganizes the CSV that way.


In [4]:
import load_to_sql
load_to_sql.main()


Loaded database at /home/user/sales-analytics-dashboard/data/processed/sales.db
  customers: 817 rows
  products: 20 rows
  orders: 7,049 rows
  order_items: 12,000 rows


## Step 3: Headline KPIs

Now that the data's in a database, let's answer the basic business
questions: how much did we make, how many orders, etc.


In [5]:
DB_PATH = PROJECT_ROOT / "data" / "processed" / "sales.db"
conn = sqlite3.connect(DB_PATH)

kpis = pd.read_sql("""
    SELECT
        ROUND(SUM(oi.net_amount), 2) AS total_revenue,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        ROUND(SUM(oi.net_amount) * 1.0 / COUNT(DISTINCT oi.order_id), 2) AS avg_order_value,
        COUNT(DISTINCT o.customer_id) AS unique_customers
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
""", conn)
kpis


,total_revenue,total_orders,avg_order_value,unique_customers
0,2627892.03,7049,372.8,817


In [6]:
monthly = pd.read_sql("""
    SELECT strftime('%Y-%m-01', o.order_date) AS month, SUM(oi.net_amount) AS revenue
    FROM order_items oi JOIN orders o ON o.order_id = oi.order_id
    GROUP BY month ORDER BY month
""", conn, parse_dates=["month"])

fig = px.line(monthly, x="month", y="revenue", markers=True, title="Monthly Revenue")
fig.show()


In [7]:
by_region = pd.read_sql("""
    SELECT c.region, ROUND(SUM(oi.net_amount), 2) AS revenue
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.region ORDER BY revenue DESC
""", conn)

fig = px.pie(by_region, values="revenue", names="region", hole=0.4, title="Revenue by Region")
fig.show()


## Step 4: Forecast the next 6 months

This looks at the seasonal pattern in the monthly revenue above (notice the
December spikes?) and extends it forward using a method called
**Holt-Winters** — it separates the data into a trend and a repeating
seasonal wave, then projects both into the future.


In [8]:
import forecasting
forecasting.main()


Wrote 42 rows (36 actual, 6 forecast) to /home/user/sales-analytics-dashboard/data/processed/revenue_forecast.csv

Forecast:
     month      revenue
2026-01-01 74688.083347
2026-02-01 69141.178351
2026-03-01 80498.124950
2026-04-01 84916.036142
2026-05-01 89201.599655
2026-06-01 97582.390797


In [9]:
forecast_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "revenue_forecast.csv", parse_dates=["month"])

fig = px.line(forecast_df, x="month", y="revenue", color="type", markers=True,
              title="Revenue: Actual vs. 6-Month Forecast")
fig.show()


## Step 5: Group customers by buying behavior (RFM segmentation)

Every customer gets scored on **R**ecency (days since last order —
lower is better), **F**requency (number of orders — higher is better), and
**M**onetary (total spend — higher is better). A clustering algorithm then
groups customers with similar scores, so instead of looking at 800+
individual people, we get a handful of meaningful groups.


In [10]:
import segmentation
segmentation.main()


Wrote 817 customer segments to /home/user/sales-analytics-dashboard/data/processed/customer_segments.csv

                 customers  avg_recency_days  avg_frequency  avg_monetary
segment                                                                  
Champions              239              40.8           17.0        6602.4
Loyal Mid-Value        362             110.3            6.9        2493.9
At Risk                139             206.0            2.6         746.0
New / Low-Value         77             661.5            1.6         563.8


In [11]:
segments_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "customer_segments.csv")

fig = px.pie(segments_df["segment"].value_counts().reset_index(),
             values="count", names="segment", hole=0.4, title="Customers by Segment")
fig.show()


In [12]:
fig = px.scatter(segments_df, x="frequency", y="monetary", color="segment",
                  size="monetary", log_y=True, hover_data=["customer_id", "recency_days"],
                  title="Customers: Frequency vs. Spend, colored by segment")
fig.show()


In [13]:
segments_df.sort_values("monetary", ascending=False).head(10)


,customer_id,recency_days,frequency,monetary,segment
773,CUST-00804,84,33,16649.46,Champions
50,CUST-00054,5,28,15081.32,Champions
134,CUST-00138,27,25,15017.12,Champions
60,CUST-00064,13,24,14945.80,Champions
388,CUST-00404,3,26,14741.56,Champions
12,CUST-00013,13,25,12219.19,Champions
332,CUST-00345,19,24,12201.25,Champions
613,CUST-00636,26,25,11404.18,Champions
635,CUST-00659,15,22,11117.92,Champions
809,CUST-00843,14,24,11067.30,Champions


## What's next?

This notebook covers the *analysis* side. The project also has a live,
interactive **dashboard** (`dashboard/app.py`) with filters and KPI cards —
that one has to be launched separately with:

```
streamlit run dashboard/app.py
```

because Streamlit runs as its own local web app rather than inside a
notebook. Everything you explored above (KPIs, forecast, segments) is what
powers that dashboard's charts.
